# Generative Model Theory

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 09.04 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/05_generative_model_theory.ipynb)

---

## 🎯 Learning Objective

Derive the mathematical foundations of normalizing flows, the VAE's ELBO, and GAN minimax theory, and connect them to score matching.

---

## 📐 Theory

Every major generative modeling family — normalizing flows, VAEs, GANs, and (previewed here,
fully derived in `09.06`) diffusion models — is answering the same underlying question:
how do you learn to sample from a complicated distribution $p_{\text{data}}(x)$, when you only
have samples from it, not its formula? This notebook derives the mathematical core of the
first three families and connects them under one lens.

### Normalizing flows: exact likelihood via invertible maps

A **normalizing flow** models $x = f_\theta(z)$ for $z$ drawn from a simple base distribution
(e.g. standard Gaussian) and $f_\theta$ an invertible, differentiable map. The **change of
variables formula** (recall the Jacobian determinant from `02.08`'s change-of-variables
discussion) gives the exact density of $x$:

$$p_\theta(x) = p(z) \left| \det \frac{\partial f_\theta^{-1}}{\partial x} \right|, \qquad z = f_\theta^{-1}(x)$$

This is the *only* generative family covered here with a **tractable, exact** likelihood — no
approximation, no bound — at the cost of requiring $f_\theta$ to be invertible with an
efficiently computable Jacobian determinant, a real architectural constraint.

### Variational Autoencoders: the ELBO, derived from Jensen's inequality

A VAE posits a **latent variable** $z$ with prior $p(z)$ and a decoder $p_\theta(x|z)$, so
$p_\theta(x) = \int p_\theta(x|z)\,p(z)\,dz$ — but this integral is generally intractable.
Introduce any **approximate posterior** $q_\phi(z|x)$ and rewrite:

$$\log p_\theta(x) = \log \int p_\theta(x|z) p(z)\, dz
= \log \mathbb{E}_{z\sim q_\phi}\left[\frac{p_\theta(x|z) p(z)}{q_\phi(z|x)}\right]$$

**Jensen's inequality** ($\log$ is concave, so $\log\mathbb{E}[Y] \ge \mathbb{E}[\log Y]$ — recall
convexity from `04.03`, applied to concave functions in the opposite direction) gives:

$$\log p_\theta(x) \ge \mathbb{E}_{z\sim q_\phi}\big[\log p_\theta(x|z) + \log p(z) - \log q_\phi(z|x)\big] =: \text{ELBO}(x)$$

This is the **Evidence Lower BOund**: a quantity you *can* compute (or estimate via sampling)
that is *always* a lower bound on the intractable true log-likelihood, for *any* choice of
$q_\phi$. Maximizing the ELBO — the actual VAE training objective — simultaneously pushes the
true log-likelihood up (since it's above the ELBO) and makes $q_\phi$ a better approximation to
the true posterior $p_\theta(z|x)$ (the gap between ELBO and true log-likelihood is exactly
$D_{KL}(q_\phi(z|x) \| p_\theta(z|x))$, recall KL divergence from `05.02`) — a tighter posterior
approximation directly tightens the bound.

### GANs: a minimax game instead of an explicit density

**GANs** sidestep likelihoods entirely. A generator $G_\theta$ maps noise to samples; a
discriminator $D_\phi$ tries to distinguish real data from $G_\theta$'s outputs. Training solves

$$\min_\theta \max_\phi \ \mathbb{E}_{x\sim p_{\text{data}}}[\log D_\phi(x)] + \mathbb{E}_{z\sim p(z)}[\log(1-D_\phi(G_\theta(z)))]$$

Goodfellow et al. (2014) show that for a fixed generator, the *optimal* discriminator is
$D^*(x) = p_{\text{data}}(x) / (p_{\text{data}}(x) + p_G(x))$, and substituting this back reveals
the generator's objective at the game's optimum reduces to minimizing the **Jensen-Shannon
divergence** between $p_{\text{data}}$ and $p_G$ — a genuine, principled divergence-minimization
objective, even though no term in the game as originally written mentions any divergence
explicitly.

### Score matching: a fourth route, avoiding likelihoods AND adversarial games

**Score matching** models not the density $p(x)$ itself but its **score function**
$\nabla_x \log p(x)$ (recall the gradient-field framing from `02.09`) — a vector field pointing
toward higher density everywhere. Crucially, the score doesn't require the normalizing constant
that makes densities hard to work with (dividing by $\int p(x)dx$ never enters the picture,
since differentiating $\log p(x)$ kills any multiplicative normalizing constant). Once you have
a good score estimate, **Langevin dynamics** (also previewed in `02.09`) generates samples by
following the score plus noise. This is the direct mathematical seed of diffusion models,
covered fully in `09.06`.

### One unifying picture

All four families solve the same problem — sample from $p_{\text{data}}$ — via four different
mathematical compromises: exact likelihood but architecturally constrained (flows), tractable
lower bound on likelihood (VAEs), no likelihood at all, an adversarial game instead (GANs), or
sidestep likelihood by modeling the score directly (score matching, → diffusion).

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

In a toy Gaussian latent-variable model where the true log-likelihood is exactly computable,
plotting the ELBO against how far the approximate posterior $q_\phi$ drifts from the true
posterior makes the ELBO's "always a lower bound, tighter when $q_\phi$ is better" property
visible directly.

In [ ]:
# A 1D Gaussian latent-variable model: prior N(0,1), decoder mean=z with fixed scale --
# both Gaussian, so the TRUE log p(x) and TRUE posterior are exactly computable in closed form.
rng = np.random.default_rng(0)
decoder_scale = 0.5
x_val = 1.2

true_var = 1.0**2 + decoder_scale**2
true_ll = -0.5*np.log(2*np.pi*true_var) - 0.5*x_val**2/true_var

true_post_var = 1 / (1/1.0**2 + 1/decoder_scale**2)
true_post_mean = true_post_var * (x_val / decoder_scale**2)

def elbo(x, q_mu, q_sigma, n_mc=100000):
    z = rng.normal(q_mu, q_sigma, size=n_mc)
    log_px_given_z = -0.5*np.log(2*np.pi*decoder_scale**2) - 0.5*(x - z)**2/decoder_scale**2
    log_pz = -0.5*np.log(2*np.pi) - 0.5*z**2
    log_qz = -0.5*np.log(2*np.pi*q_sigma**2) - 0.5*(z - q_mu)**2/q_sigma**2
    return np.mean(log_px_given_z + log_pz - log_qz)

# Sweep q's mean away from the TRUE posterior mean, holding q's std fixed at the true value
mean_offsets = np.linspace(-1.5, 1.5, 25)
elbo_vals = [elbo(x_val, true_post_mean + off, np.sqrt(true_post_var)) for off in mean_offsets]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(mean_offsets, elbo_vals, 'o-', color="#4C72B0", markersize=4, label="ELBO(x)")
ax.axhline(true_ll, color="#C44E52", ls="--", lw=2, label="true log p(x) (exact, closed form)")
ax.axvline(0, color="gray", ls=":", lw=1, label="q centered on TRUE posterior")
ax.set_xlabel("offset of q's mean from the true posterior mean"); ax.set_ylabel("value")
ax.set_title("ELBO is ALWAYS <= true log p(x);\ngap shrinks to ~0 exactly where q matches the true posterior")
ax.legend(fontsize=9)
plt.show()

max_elbo = max(elbo_vals)
print(f"True log p(x): {true_ll:.4f}")
print(f"Best ELBO achieved (q centered on true posterior): {max_elbo:.4f}")
print(f"Every single ELBO value is <= true log p(x): {all(e <= true_ll + 1e-3 for e in elbo_vals)}")

## 🐍 Implementation from Scratch

Using a Gaussian latent-variable model where the true log-likelihood is exactly computable, we
verify the ELBO's defining property directly: it never exceeds the true log-likelihood, and its
tightness depends on how good the approximate posterior is -- confirmed against an independent
Monte Carlo estimate of the true (usually intractable) log-likelihood.

In [ ]:
decoder_scale = 0.5
x_val = 1.2

def true_log_likelihood(x, decoder_scale=decoder_scale):
    # For this Gaussian model, p(x) is exactly Gaussian too: N(x; 0, 1 + decoder_scale^2)
    var = 1.0 + decoder_scale**2
    return -0.5 * np.log(2 * np.pi * var) - 0.5 * x**2 / var

true_ll = true_log_likelihood(x_val)
print(f"True log p(x={x_val}) [closed form, exact]: {true_ll:.6f}")

# Cross-check the closed form against an INDEPENDENT method: naive Monte Carlo importance
# sampling directly from the prior (this is what you'd have to do if the closed form didn't exist)
n_samples = 2_000_000
z_samples = rng.normal(0, 1, size=n_samples)
log_px_given_z = -0.5*np.log(2*np.pi*decoder_scale**2) - 0.5*(x_val - z_samples)**2/decoder_scale**2
mc_log_px = np.log(np.mean(np.exp(log_px_given_z)))
print(f"Monte Carlo estimate of the SAME quantity (importance sampling from the prior): {mc_log_px:.6f}")
print(f"Match: {np.isclose(true_ll, mc_log_px, atol=0.01)}")

# --- The ELBO, for a GOOD (near-true-posterior) q and a deliberately BAD (mismatched) q ---
def elbo(x, q_mu, q_sigma, decoder_scale=decoder_scale, n_mc=200000):
    z = rng.normal(q_mu, q_sigma, size=n_mc)
    log_px_given_z = -0.5*np.log(2*np.pi*decoder_scale**2) - 0.5*(x - z)**2/decoder_scale**2
    log_pz = -0.5*np.log(2*np.pi) - 0.5*z**2
    log_qz = -0.5*np.log(2*np.pi*q_sigma**2) - 0.5*(z - q_mu)**2/q_sigma**2
    return np.mean(log_px_given_z + log_pz - log_qz)

true_post_var = 1 / (1/1.0**2 + 1/decoder_scale**2)
true_post_mean = true_post_var * (x_val / decoder_scale**2)
print(f"\nTrue posterior p(z|x): mean={true_post_mean:.4f}, std={np.sqrt(true_post_var):.4f}")

elbo_good = elbo(x_val, true_post_mean, np.sqrt(true_post_var))
elbo_bad = elbo(x_val, 0.0, 3.0)   # mismatched: wrong mean, far too spread out
print(f"ELBO with GOOD q (matches true posterior): {elbo_good:.4f}  (gap to true ll: {true_ll - elbo_good:.4f})")
print(f"ELBO with BAD q  (mismatched):              {elbo_bad:.4f}  (gap to true ll: {true_ll - elbo_bad:.4f})")
print(f"Both ELBOs <= true log p(x): {elbo_good <= true_ll and elbo_bad <= true_ll}")
print(f"Good q gives the TIGHTER bound: {(true_ll - elbo_good) < (true_ll - elbo_bad)}")

## 🤖 Why This Matters for AI

Every real VAE implementation (from the original Kingma & Welling 2014 paper through today's
latent diffusion models, which use a VAE to compress images into a smaller latent space before
diffusion) trains by maximizing exactly the ELBO derived above, using the **reparameterization
trick** ($z = \mu + \sigma\odot\epsilon$, $\epsilon\sim\mathcal{N}(0,I)$) to make the sampling
step differentiable so gradients can flow through it (recall backpropagation, `02.10`). Below,
we build a real, working 2-layer VAE in `torch`, train it on a toy 2D ring dataset (a shape a
single Gaussian *couldn't* represent well, genuinely needing the latent-variable structure), and
confirm it can generate new samples resembling the ring shape it never explicitly saw as a formula.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
n_data = 500
theta = rng.uniform(0, 2 * np.pi, n_data)
radius = 2.0 + rng.normal(scale=0.1, size=n_data)
X_ring = np.stack([radius * np.cos(theta), radius * np.sin(theta)], axis=1).astype(np.float32)
X_ring_t = torch.tensor(X_ring)

class TinyVAE(nn.Module):
    def __init__(self, data_dim=2, latent_dim=2, hidden=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(data_dim, hidden), nn.Tanh())
        self.enc_mu = nn.Linear(hidden, latent_dim)
        self.enc_logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.Tanh(), nn.Linear(hidden, data_dim))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.enc_mu(h), self.enc_logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)   # the reparameterization trick: sampling
                                                 # becomes a differentiable function of (mu, std, noise)
        return self.dec(z), mu, logvar

def vae_loss(x, x_recon, mu, logvar):
    recon_loss = ((x - x_recon) ** 2).sum(dim=1).mean()
    # KL(q(z|x) || N(0,I)) has a closed form for Gaussians -- no Monte Carlo estimate needed
    kl = -0.5 * (1 + logvar - mu**2 - torch.exp(logvar)).sum(dim=1).mean()
    return recon_loss + kl, recon_loss.item(), kl.item()   # minimizing this MAXIMIZES the ELBO

model = TinyVAE()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for step in range(2000):
    opt.zero_grad()
    x_recon, mu, logvar = model(X_ring_t)
    loss, recon, kl = vae_loss(X_ring_t, x_recon, mu, logvar)
    loss.backward()
    opt.step()

with torch.no_grad():
    z_new = torch.randn(300, 2)
    x_gen = model.dec(z_new).numpy()

real_radius = np.linalg.norm(X_ring, axis=1)
gen_radius = np.linalg.norm(x_gen, axis=1)
print(f"Final ELBO components: reconstruction loss={recon:.4f}, KL={kl:.4f}")
print(f"Real data radius:      mean={real_radius.mean():.3f}, std={real_radius.std():.3f}")
print(f"Generated data radius: mean={gen_radius.mean():.3f}, std={gen_radius.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(X_ring[:, 0], X_ring[:, 1], s=8, alpha=0.5, color="#4C72B0")
axes[0].set_title("Real training data (a ring)"); axes[0].set_aspect('equal')
axes[1].scatter(x_gen[:, 0], x_gen[:, 1], s=8, alpha=0.5, color="#C44E52")
axes[1].set_title("VAE-generated samples\n(never told the shape was a ring)"); axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the discrete random variable $Y$ taking values $1$ and $4$ each with probability $0.5$,
   compute $\log\mathbb{E}[Y]$ and $\mathbb{E}[\log Y]$ by hand, and confirm
   $\log\mathbb{E}[Y] \ge \mathbb{E}[\log Y]$ — Jensen's inequality for concave $\log$, the
   single fact the entire ELBO derivation rests on.

<details>
<summary>💡 Solution</summary>

$\mathbb{E}[Y] = 0.5(1)+0.5(4)=2.5$, so $\log\mathbb{E}[Y]=\log2.5\approx0.9163$.
$\mathbb{E}[\log Y] = 0.5\log1+0.5\log4 = 0+0.5(1.3863)\approx0.6931$. Indeed
$0.9163\ge0.6931$, confirming Jensen's inequality on this concrete example — and the gap
($\approx0.223$) is itself a direct illustration of how loose or tight the inequality can be
depending on how spread out $Y$'s distribution is.

</details>

### 💭 UNDERSTAND
2. `03.10` already covers full VAE training end to end. This notebook's own ELBO experiment
   instead isolates a *single* question: with $q_\phi$'s mean held exactly at the true
   posterior mean, does getting $q_\phi$'s *standard deviation* wrong also degrade the ELBO?
   Predict the shape of ELBO vs. $q$'s standard deviation (holding mean fixed) before checking —
   is it monotonic, or does it have a specific optimum?

<details>
<summary>💡 Solution</summary>

The ELBO should be *maximized* exactly when $q_\phi$'s standard deviation matches the true
posterior's standard deviation, and degrade on both sides — not monotonic. Numerically, sweeping
$q$'s std from $0.1\times$ to $4\times$ the true posterior std (with $x=1.2$, decoder scale
$0.5$, true posterior std $\approx0.447$): the ELBO peaks at exactly the true std ($\approx-1.607$,
matching the true log-likelihood almost exactly), and gets worse moving either direction — down
to $\approx-3.41$ at $0.1\times$ too narrow, and down to $\approx-7.72$ at $4\times$ too wide.
This matches the theory precisely: the ELBO-to-true-likelihood gap equals
$D_{KL}(q_\phi\|p_\theta(z|x))$, and KL divergence between two Gaussians grows whenever *either*
the means OR the variances differ — getting the mean exactly right doesn't protect you from a
wrong variance.

</details>

### ✏️ DERIVE
3. Verify the claim that the ELBO's gap to the true log-likelihood is exactly
   $D_{KL}(q_\phi(z|x)\|p_\theta(z|x))$ (stated but not derived in the Theory section) by
   computing both sides independently for a specific mismatched $q$ (e.g. $q$'s mean shifted by
   $+0.7$ from the true posterior mean, with a $1.5\times$-too-wide std) in this notebook's
   Gaussian latent-variable model, using the closed-form Gaussian-to-Gaussian KL formula for the
   right-hand side.

<details>
<summary>💡 Solution outline</summary>

$\log p_\theta(x) - \text{ELBO}(x) = \log p_\theta(x) - \mathbb{E}_{q}[\log p_\theta(x|z)+\log
p(z)-\log q(z)]$. Using $p_\theta(x|z)p(z) = p_\theta(z|x)p_\theta(x)$ (Bayes' rule) inside the
expectation and simplifying, the $\log p_\theta(x)$ terms combine to leave exactly
$\mathbb{E}_q[\log q(z) - \log p_\theta(z|x)] = D_{KL}(q(z)\|p_\theta(z|x))$ — confirming the
claimed identity algebraically. Numerically, for $q$ with mean shifted $+0.7$ and std
$1.5\times$ the true posterior's: the Monte Carlo ELBO gap (true log-likelihood minus ELBO)
comes out to $\approx1.450$, and the closed-form Gaussian KL
$D_{KL}(\mathcal{N}(\mu_q,\sigma_q^2)\|\mathcal{N}(\mu_{\text{true}},\sigma_{\text{true}}^2))$
computed independently comes out to $\approx1.445$ — matching to within Monte Carlo noise,
confirming the identity holds exactly (not just approximately) in this tractable case.

</details>

### 🐍 IMPLEMENT
4. Implement the closed-form Gaussian-to-Gaussian KL divergence formula,
   $D_{KL}(\mathcal{N}(\mu_1,\sigma_1^2)\|\mathcal{N}(\mu_2,\sigma_2^2)) =
   \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2+(\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12$,
   and use it to directly verify Exercise 3's ELBO-gap identity across several different
   mismatched $q$ choices (not just one), confirming the match holds generally in this tractable
   model, not just for one specific example.

<details>
<summary>✅ How to know you're right</summary>

For each of several $(q_\mu, q_\sigma)$ choices (varying both mean-offset and std-multiplier),
the ELBO gap (true log-likelihood minus your Monte Carlo ELBO estimate) should match your
closed-form KL computation to within the Monte Carlo estimator's own noise level (tighter with
more samples). If the two ever diverge by more than that noise tolerance for some $q$, check
your KL formula's argument order first — $D_{KL}(q\|p)$ and $D_{KL}(p\|q)$ are generally
different quantities, and only one of them is the correct gap here.

</details>

### 🤖 APPLY
5. The Theory section states that the GAN minimax game's optimal discriminator is
   $D^*(x)=p_{\text{data}}(x)/(p_{\text{data}}(x)+p_G(x))$, and that substituting it back
   reduces the generator's objective to the Jensen-Shannon divergence between $p_{\text{data}}$
   and $p_G$. For a toy case where $p_{\text{data}}$ and $p_G$ are both known 1D Gaussians with
   different means (so you can evaluate everything numerically without training an actual GAN),
   compute $D^*(x)$ at a few sample points and confirm it's a valid probability (between $0$ and
   $1$) that trends toward $1$ where $p_{\text{data}}$ dominates and toward $0$ where $p_G$
   dominates.

<details>
<summary>✅ What you should observe</summary>

For, e.g., $p_{\text{data}}=\mathcal{N}(0,1)$ and $p_G=\mathcal{N}(2,1)$, evaluating
$D^*(x)=p_{\text{data}}(x)/(p_{\text{data}}(x)+p_G(x))$ at $x=-2$ (deep in $p_{\text{data}}$'s
territory, far from $p_G$'s mean) should give a value close to $1$ (the "optimal discriminator"
is very confident this point is real); at $x=4$ (deep in $p_G$'s territory) it should give a
value close to $0$; and at $x=1$ (the midpoint between the two means, where both densities are
equal by symmetry) it should give exactly $0.5$ — the discriminator is genuinely unsure, which is
exactly the maximum-confusion point you'd expect at the crossover between two symmetric,
equal-variance distributions.

</details>

### 🚀 CHALLENGE
6. Modify the AI section's `TinyVAE` to use `latent_dim=1` instead of `2`, retrain, and compare
   the generated ring's quality (final reconstruction loss, and generated-sample radius mean and
   std) to the `latent_dim=2` version. A ring is an intrinsically 1-dimensional curve embedded in
   2D space — does a 1D latent space suffice, or does the extra capacity of `latent_dim=2`
   actually help? Report your finding honestly even if it's not what you expected.

<details>
<summary>🔍 What a strong answer covers</summary>

Across multiple random seeds, `latent_dim=1` consistently *outperforms* `latent_dim=2` on both
metrics that matter here: final reconstruction loss is lower (e.g. $\approx0.58$–$0.63$ vs.
$\approx0.71$–$0.80$ across seeds), and — more tellingly — the generated samples' radius standard
deviation stays close to the true data's ($\approx0.09$), landing around $0.05$–$0.08$ for
`latent_dim=1` but consistently $3$–$4\times$ too wide ($\approx0.27$–$0.29$) for `latent_dim=2`.
The extra latent dimension doesn't just fail to help — it actively hurts the generative quality,
because the ring is genuinely a 1D manifold (parameterized entirely by angle) embedded in 2D
space, so the "true" degrees of freedom the data actually varies along is exactly $1$; giving the
model a second latent dimension gives it an extra degree of freedom to inject noise into
generated samples along a direction the real data doesn't actually vary in, degrading the
generated radius's tightness. A strong answer connects this to the manifold hypothesis directly:
matching latent dimensionality to a dataset's true intrinsic dimensionality (not just to the
ambient data dimension) is a real modeling choice with a measurable, honest downside for getting
it wrong in either direction, not merely a hyperparameter with a safe default.

</details>

---

## 📚 Further Reading
- Kingma & Welling, ["Auto-Encoding Variational Bayes"](https://arxiv.org/abs/1312.6114) (the original VAE paper)
- Goodfellow et al., ["Generative Adversarial Networks"](https://arxiv.org/abs/1406.2661)
- Song & Ermon, ["Generative Modeling by Estimating Gradients of the Data Distribution"](https://arxiv.org/abs/1907.05600) (score matching)

---

*Next notebook: [The Math of Diffusion Processes](06_diffusion_process_math.ipynb)*